# Part 2c: AppCon Uncertainties and Export to SimPEG and UBCGIF Convention

**WHAT THE NOTEBOOK DOES:**

1. Loads the sorted data file (n_freq, n_comp, n_loc)
2. Defines user-specified uncertainties
3. Uses the uncertainties to add Gaussian noise to the data
4. Outputs true data, observed data and uncertainties

**Data Conversions:** Unnecessary since data are scalar and already S/m.

In [1]:
from simpeg.electromagnetics import natural_source as nsem
from simpeg.utils import mkvc, ndgrid, plot2Ddata

# Basic Python functionality
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FormatStrFormatter

mpl.rcParams.update({"font.size": 14})

from ipywidgets import (
    interact,
    interactive,
    IntSlider,
    widget,
    FloatText,
    FloatSlider,
    fixed,
)

## Load the Data

In [2]:
in_dir = './part_1_outputs/'
out_dir = './part_2_outputs/'

if not os.path.exists(out_dir):
    os.mkdir(out_dir)

In [3]:
frequencies = np.load(in_dir + 'frequencies_appcon.npy')
locations = np.load(in_dir + 'locations_appcon.npy')
dtrue = np.load(in_dir + 'data_appcon.npy')

n_freq = np.shape(dtrue)[0]
n_loc = np.shape(dtrue)[1]

## Assign Uncertainties

Here, we assign uncertainties equal to a fraction of the large amplitude anomaly relative to the median value. We do this independently for each frequency.

In [4]:
frac = 0.1

In [5]:
unc = np.zeros_like(dtrue)

for jj in range(n_freq):
    d_temp = dtrue[jj, :]
    unc[jj, :] = frac * np.max(np.abs(d_temp - np.median(d_temp)))

noise = unc * np.random.normal(size=(n_freq, n_loc))
dobs = dtrue + noise

## Plot

In [6]:
mpl.rcParams.update({'font.size': 13})

mpl.rcParams.update({'font.size': 13})

d_list = [dtrue, dobs, noise]

def plot_data(f_ind):
    
    f_ind = f_ind - 1

    fig = plt.figure(figsize=(12, 4))

    ax1 = 3 * [None]
    ax2 = 3 *[None]
    norm = 3 * [None]
    cs = 3 * [None]
    cplot = 3 * [None]
    cbar = 3 * [None]
    cmap = 3 * [None]

    COUNT = 0

    for ii in range(3):  # true, obs, noise

        ax1[COUNT] = fig.add_axes([0.05+0.3*ii, 0.25, 0.25, 0.7])
        ax2[COUNT] = fig.add_axes([0.07+0.3*ii, 0.05, 0.21, 0.05])

        if ii == 0:
            data_temp = d_list[ii][f_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtrue' 
        elif ii == 1:
            data_temp = d_list[ii][f_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dobs'
        if ii == 2:
            data_temp = d_list[ii][f_ind, :]
            cmap[COUNT] = mpl.cm.RdBu_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'noise'

        norm[COUNT]= mpl.colors.Normalize(vmin=vmin, vmax=vmax)

        cplot[COUNT], ax1[COUNT] = plot2Ddata(
            locations[:, 0:2],
            data_temp,
            nx=200,
            ny=200,
            ax=ax1[COUNT],
            ncontour=200,
            contourOpts={"cmap": cmap[COUNT], "norm": norm[COUNT]},
        )

        ax1[COUNT].set_title(title + ': {} Hz'.format(frequencies[f_ind]))
        ax1[COUNT].set_xlabel('Easting (m)')
        if ii == 0:
            ax1[COUNT].set_ylabel('Northing (m)')
        else:
            ax1[COUNT].set_yticks([])

        cbar[COUNT]= mpl.colorbar.ColorbarBase(ax2[COUNT], norm=norm[COUNT], orientation="horizontal", cmap=cmap[COUNT])
        cbar[COUNT].set_label('S/m')
        ax2[COUNT].set_xticks([vmin, 0.5*(vmin+vmax), vmax])
        
        COUNT = COUNT + 1

def DataWidget():

    i = interact(
        plot_data,
        f_ind=IntSlider(
            min=1,
            max=n_freq,
            value=1,
            step=1,
            continuous_update=False,
            description="FREQID",
        ),
    )
    
    return i

In [7]:
DataWidget()

interactive(children=(IntSlider(value=1, continuous_update=False, description='FREQID', max=22, min=1), Output…

<function __main__.plot_data(f_ind)>

In [8]:
frequencies

array([   25.10002,    31.59997,    39.80004,    50.09995,    63.09984,
          79.39974,   100.     ,   125.9    ,   158.5    ,   199.5001 ,
         251.2001 ,   316.1995 ,   398.1003 ,   501.2003 ,   630.9985 ,
         794.3001 ,  5011.903  ,  6309.586  ,  7943.317  , 10000.     ,
       12589.19   , 15848.89   ])

## Output Data

In [9]:
np.save(out_dir + 'dtrue_appcon.npy', dtrue)
np.save(out_dir + 'dobs_appcon.npy', dobs)
np.save(out_dir + 'unc_appcon.npy', unc)